<a href="https://colab.research.google.com/github/yalama34/coursearch/blob/research%2FCatBoost/CatBoost_Model_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CatBoost Model Pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import math
import os

DATA_DIR = "/content/drive/MyDrive/coursearch/"


with open(os.path.join(DATA_DIR, 'courses.json'), 'r', encoding='utf-8') as f:
    courses_data = json.load(f)['courses']
courses_df = pd.DataFrame(courses_data)

with open(os.path.join(DATA_DIR, 'users.json'), 'r', encoding='utf-8') as f:
    users_data = json.load(f)['users']
users_df = pd.DataFrame(users_data)

with open(os.path.join(DATA_DIR, 'junction_seed.json'), 'r', encoding='utf-8') as f:
    junction = json.load(f)

with open(os.path.join(DATA_DIR, 'actions_seed.json'), 'r', encoding='utf-8') as f:
    actions_data = json.load(f)

actions_df = pd.DataFrame(actions_data['actions'])
actions_df['timestamp'] = pd.to_datetime(actions_df['timestamp'])
actions_df = actions_df.sort_values('timestamp').reset_index(drop=True)

print(f"Событий: {len(actions_df)}")
print(f"Уникальных пользователей: {actions_df['user_temp_id'].nunique()}")
print(f"Уникальных курсов: {actions_df['course_temp_id'].nunique()}")

Событий: 24607
Уникальных пользователей: 96
Уникальных курсов: 42


In [ ]:
course_info = {}
for _, row in courses_df.iterrows():
    course_info[row['temp_id']] = {
        'difficulty': row['difficulty'],
        'age_days': row['age_days'],
        'tags': set()
    }

user_info = {}
for _, row in users_df.iterrows():
    user_info[row['temp_id']] = {
        'preferred_difficulty': row['preferred_difficulty'],
        'tags': set()
    }

for pair in junction['course_tag_pairs']:
    cid = pair['course_temp_id']
    if cid in course_info:
        course_info[cid]['tags'].add(pair['tag_id'])

for pair in junction['user_tag_pairs']:
    uid = pair['user_temp_id']
    if uid in user_info:
        user_info[uid]['tags'].add(pair['tag_id'])


for cid in course_info:
    course_info[cid]['tags_count'] = len(course_info[cid]['tags'])
for uid in user_info:
    user_info[uid]['tags_count'] = len(user_info[uid]['tags'])

## Разделим на 2 датафрейма - просмотры и лайки

In [ ]:
views = actions_df[actions_df['action_type'] == 'view'].copy()
likes = actions_df[actions_df['action_type'] == 'like'][['user_temp_id', 'course_temp_id', 'timestamp']]

def find_like_within_window(view_row, likes_df, window_hours=4):
    user = view_row['user_temp_id']
    course = view_row['course_temp_id']
    view_time = view_row['timestamp']
    mask = (likes_df['user_temp_id'] == user) & \
           (likes_df['course_temp_id'] == course) & \
           (likes_df['timestamp'] > view_time) & \
           (likes_df['timestamp'] <= view_time + pd.Timedelta(hours=window_hours))
    return 1 if mask.any() else 0

views['target'] = views.apply(lambda row: find_like_within_window(row, likes), axis=1)
print(f"Target distribution:\n{views['target'].value_counts(normalize=True)}")

Target distribution:
target
0    0.813949
1    0.186051
Name: proportion, dtype: float64


In [ ]:
def compute_course_popularity_weight(
    views_count: int,
    likes_count: int,
    created_at: datetime,
    current_time: datetime,
    weights: dict = None,
    half_life_days: int = 90,
    scale: int = 1000
) -> float:
    if weights is None:
        weights = {'views': 0.2, 'likes': 0.5, 'novelty': 0.3}
    age_days = (current_time - created_at).days
    if age_days <= 0:
        novelty_score = scale
    else:
        decay = math.exp(-age_days * 0.693147 / half_life_days)
        novelty_score = decay * scale
    total = (views_count * weights['views'] +
             likes_count * weights['likes'] +
             novelty_score * weights['novelty'])
    return round(total, 4)

In [ ]:
user_views = {}      # uid -> list of timestamps
user_likes = {}      # uid -> list of timestamps
course_views = {}    # cid -> list of timestamps
course_likes = {}    # cid -> list of timestamps
user_course_views = {}  # uid -> dict cid -> count
user_course_likes = {}  # uid -> set of cid
user_views_history = {}  # uid -> list of (ts, cid, tags)

features_list = []

target_map = {}
for _, row in views.iterrows():
    target_map[(row['user_temp_id'], row['course_temp_id'], row['timestamp'])] = row['target']

def count_window(event_list, current_ts, days):
    if not event_list:
        return 0
    cutoff = current_ts - timedelta(days=days)
    cnt = 0
    for t in reversed(event_list):
        if t >= cutoff and t < current_ts:
            cnt += 1
        elif t < cutoff:
            break
    return cnt


for idx, event in actions_df.iterrows():
    uid = event['user_temp_id']
    cid = event['course_temp_id']
    ts = event['timestamp']
    action = event['action_type']

    if action == 'view':
        u_info = user_info.get(uid, {})
        c_info = course_info.get(cid, {})

        u_tags = u_info.get('tags', set())
        c_tags = c_info.get('tags', set())
        common = len(u_tags & c_tags)
        total_tags = len(u_tags) + len(c_tags)
        jaccard = common / max(1, total_tags - common)

        user_tags_cnt = len(u_tags)
        course_tags_cnt = len(c_tags)

        user_diff = u_info.get('preferred_difficulty')
        course_diff = c_info.get('difficulty')
        diff_match = 1 if (user_diff and user_diff == course_diff) else 0

        user_domain = u_info.get('primary_domain')
        course_domain = c_info.get('primary_domain')
        domain_match = 1 if (user_domain and user_domain == course_domain) else 0

        user_views_7d = count_window(user_views.get(uid, []), ts, 7)
        user_views_30d = count_window(user_views.get(uid, []), ts, 30)
        user_likes_7d = count_window(user_likes.get(uid, []), ts, 7)
        user_likes_30d = count_window(user_likes.get(uid, []), ts, 30)

        total_views = len(course_views.get(cid, []))
        total_likes = len(course_likes.get(cid, []))
        age_days = c_info.get('age_days', 0)
        created_at = ts - timedelta(days=age_days)
        pop_weight = compute_course_popularity_weight(total_views, total_likes, created_at, ts)


        if uid in user_views_history:
            cutoff = ts - timedelta(days=7)
            recent_views = [(t, c, tags) for (t, c, tags) in user_views_history[uid] if t >= cutoff]
            unique_courses_7d = len({c for (_, c, _) in recent_views})
            unique_tags_7d = len(set().union(*[tags for (_, _, tags) in recent_views]))
        else:
            unique_courses_7d = 0
            unique_tags_7d = 0

        target_val = target_map.get((uid, cid, ts), 0)

        features_list.append({
            'user_temp_id': uid,
            'course_temp_id': cid,
            'timestamp': ts,
            'user_tags_count': user_tags_cnt,
            'course_tags_count': course_tags_cnt,
            'common_tags': common,
            'jaccard_tags': jaccard,
            'difficulty_match': diff_match,
            'domain_match': domain_match,
            'user_views_7d': user_views_7d,
            'user_likes_7d': user_likes_7d,
            'user_views_30d': user_views_30d,
            'user_likes_30d': user_likes_30d,
            'course_popularity_weight': pop_weight,
            'user_unique_courses_7d': unique_courses_7d,
            'user_unique_tags_7d': unique_tags_7d,
            'target': target_val
        })

        # Обновление счётчиков
        user_views.setdefault(uid, []).append(ts)
        course_views.setdefault(cid, []).append(ts)
        user_course_views.setdefault(uid, {}).setdefault(cid, 0)
        user_course_views[uid][cid] += 1
        user_views_history.setdefault(uid, []).append((ts, cid, c_tags))

    elif action == 'like':
        user_likes.setdefault(uid, []).append(ts)
        course_likes.setdefault(cid, []).append(ts)
        user_course_likes.setdefault(uid, set()).add(cid)

features_df = pd.DataFrame(features_list)
print(f"Датасет: {features_df.shape}")

Датасет: (20747, 17)


#Загрузим косинусное расстояние

In [ ]:

import pandas as pd
import os

dist_df = pd.read_csv(os.path.join(DATA_DIR, 'cosine_distances.csv'), index_col=0)


dist_long = dist_df.stack().reset_index()
dist_long.columns = ['user_temp_id', 'course_temp_id', 'cosine_distance']

features_df = features_df.merge(
    dist_long,
    on=['user_temp_id', 'course_temp_id'],
    how='left'
)

print(f"Столбцы после добавления: {features_df.columns.tolist()}")
print(f"Пропусков cosine_distance: {features_df['cosine_distance'].isna().sum()}")


if features_df['cosine_distance'].isna().sum() > 0:
    features_df['cosine_distance'] = features_df['cosine_distance'].fillna(features_df['cosine_distance'].median())


Столбцы после добавления: ['user_temp_id', 'course_temp_id', 'timestamp', 'user_tags_count', 'course_tags_count', 'common_tags', 'jaccard_tags', 'difficulty_match', 'domain_match', 'user_views_7d', 'user_likes_7d', 'user_views_30d', 'user_likes_30d', 'course_popularity_weight', 'user_unique_courses_7d', 'user_unique_tags_7d', 'target', 'cosine_distance']
Пропусков cosine_distance: 0


#Добавим негативных сучаев

In [ ]:
import numpy as np
from datetime import timedelta

def add_negative_samples(features_df, n_negatives=3):

    user_history = {}  # uid -> list of (timestamp, course_id)
    for _, row in features_df.iterrows():
        uid = row['user_temp_id']
        ts = row['timestamp']
        cid = row['course_temp_id']
        user_history.setdefault(uid, []).append((ts, cid))

    all_courses = features_df['course_temp_id'].unique()


    cos_dist_dict = {}
    for _, row in features_df.iterrows():
        uid = row['user_temp_id']
        cid = row['course_temp_id']
        cos_dist_dict[(uid, cid)] = row['cosine_distance']

    new_rows = []
    pos_events = features_df[features_df['target'] == 1]

    for idx, pos_row in pos_events.iterrows():
        uid = pos_row['user_temp_id']
        ts = pos_row['timestamp']

        seen_courses = {cid for t, cid in user_history.get(uid, []) if t <= ts}

        candidate_courses = [c for c in all_courses if c not in seen_courses]

        if len(candidate_courses) == 0:
            continue


        n_sample = min(n_negatives, len(candidate_courses))
        neg_courses = np.random.choice(candidate_courses, n_sample, replace=False)

        for neg_cid in neg_courses:
            new_row = pos_row.copy()
            new_row['course_temp_id'] = neg_cid
            new_row['target'] = 0

            c_info = course_info.get(neg_cid, {})
            c_tags = c_info.get('tags', set())
            u_info = user_info.get(uid, {})
            u_tags = u_info.get('tags', set())


            common = len(u_tags & c_tags)
            total_tags = len(u_tags) + len(c_tags)
            new_row['jaccard_tags'] = common / max(1, total_tags - common)
            new_row['common_tags'] = common
            new_row['course_tags_count'] = len(c_tags)


            user_diff = u_info.get('preferred_difficulty')
            course_diff = c_info.get('difficulty')
            new_row['difficulty_match'] = 1 if (user_diff and user_diff == course_diff) else 0


            user_domain = u_info.get('primary_domain')
            course_domain = c_info.get('primary_domain')
            new_row['domain_match'] = 1 if (user_domain and user_domain == course_domain) else 0

            course_views_hist = course_views.get(neg_cid, [])
            course_likes_hist = course_likes.get(neg_cid, [])
            total_views = len([t for t in course_views_hist if t <= ts])
            total_likes = len([t for t in course_likes_hist if t <= ts])
            age_days = c_info.get('age_days', 0)
            created_at = ts - timedelta(days=age_days)
            new_row['course_popularity_weight'] = compute_course_popularity_weight(
                total_views, total_likes, created_at, ts
            )

            cos_dist = cos_dist_dict.get((uid, neg_cid))
            if cos_dist is None:
                user_cos_dists = [v for (u, c), v in cos_dist_dict.items() if u == uid]
                cos_dist = np.mean(user_cos_dists) if user_cos_dists else 0.5
            new_row['cosine_distance'] = cos_dist

            new_rows.append(new_row)


    features_df_augmented = pd.concat([features_df, pd.DataFrame(new_rows)], ignore_index=True)

    print(f"Добавлено {len(new_rows)} негативных сэмплов")
    print(f"Новый размер датасета: {features_df_augmented.shape}")
    print(f"Target distribution:")
    print(f"  0: {features_df_augmented['target'].value_counts(normalize=True)[0]:.4f}")
    print(f"  1: {features_df_augmented['target'].value_counts(normalize=True)[1]:.4f}")

    return features_df_augmented

features_df = add_negative_samples(features_df, n_negatives=3)

print("\nПроверка наличия обоих классов у пользователей:")
for uid in features_df['user_temp_id'].unique()[:5]:
    user_data = features_df[features_df['user_temp_id'] == uid]
    print(f"  {uid}: targets {user_data['target'].unique()}")

Добавлено 8820 негативных сэмплов
Новый размер датасета: (29567, 18)
Target distribution:
  0: 0.8694
  1: 0.1306

Проверка наличия обоих классов у пользователей:
  u_009: targets [0 1]
  u_076: targets [1 0]
  u_043: targets [0 1]
  u_045: targets [0 1]
  u_094: targets [0 1]


In [ ]:
features_df = features_df.sort_values('timestamp').reset_index(drop=True)

# Разделяем по времени (80/20)
split_idx = int(len(features_df) * 0.8)
split_date = features_df.loc[split_idx, 'timestamp']

train_df = features_df[features_df['timestamp'] < split_date].copy()
val_df = features_df[features_df['timestamp'] >= split_date].copy()

print(f"Train: {train_df.shape}, Val: {val_df.shape}")
print(f"Train target rate: {train_df['target'].mean():.4f}")
print(f"Val target rate: {val_df['target'].mean():.4f}")

for uid in val_df['user_temp_id'].unique()[:5]:
    targets = val_df[val_df['user_temp_id'] == uid]['target'].unique()
    print(f"User {uid}: targets {targets}")

Train: (23650, 18), Val: (5917, 18)
Train target rate: 0.1240
Val target rate: 0.1567
User u_096: targets [0 1]
User u_085: targets [1 0]
User u_002: targets [0 1]
User u_079: targets [0 1]
User u_057: targets [0 1]


In [ ]:
!pip install catboost
!pip install scikit-learn

#CatBoost1

In [ ]:
from catboost import CatBoostClassifier, Pool

feature_cols = [
    'user_tags_count', 'course_tags_count', 'common_tags', 'jaccard_tags',
    'difficulty_match', 'user_views_7d', 'user_likes_7d', 'user_views_30d',
    'user_likes_30d', 'course_popularity_weight',
    'user_unique_courses_7d', 'user_unique_tags_7d', 'cosine_distance'
]

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.0001,
    depth=8,
    random_seed=42,
    eval_metric='AUC',
    early_stopping_rounds=100,
    auto_class_weights='Balanced',
    verbose=100
)

model.fit(train_pool, eval_set=val_pool)

0:	test: 0.6313394	best: 0.6313394 (0)	total: 72ms	remaining: 1m 11s
100:	test: 0.6421684	best: 0.6430162 (60)	total: 4.55s	remaining: 40.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6430162374
bestIteration = 60

Shrink model to first 61 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, early_stopping_rounds=100, eval_metric='AUC', iterations=1000, learning_rate=0.0001, random_seed=42, verbose=100)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_pred_proba_cb = model.predict_proba(X_val)[:, 1]
roc_cb = roc_auc_score(y_val, y_pred_proba_cb)
pr_cb = average_precision_score(y_val, y_pred_proba_cb)

print(f"CatBoost: ROC-AUC={roc_cb:.4f}, PR-AUC={pr_cb:.4f}")

# Feature importance
importances = model.get_feature_importance(prettified=True)
print(importances)

CatBoost: ROC-AUC=0.6472, PR-AUC=0.2365
                  Feature Id  Importances
0           difficulty_match    43.198566
1                common_tags    25.077817
2               jaccard_tags    23.584435
3   course_popularity_weight     3.996200
4            cosine_distance     1.164350
5          course_tags_count     0.850607
6            user_tags_count     0.419821
7             user_likes_30d     0.408172
8              user_views_7d     0.327049
9              user_likes_7d     0.310765
10       user_unique_tags_7d     0.248004
11            user_views_30d     0.214406
12    user_unique_courses_7d     0.199808


#CatBoost2

In [ ]:
from catboost import CatBoostClassifier, Pool

feature_cols = [
    'user_tags_count', 'course_tags_count', 'common_tags', 'jaccard_tags',
    'difficulty_match', 'user_views_7d', 'user_likes_7d', 'user_views_30d',
    'user_likes_30d', 'course_popularity_weight',
    'user_unique_courses_7d', 'user_unique_tags_7d','cosine_distance'
]

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.0001,
    depth=5,
    l2_leaf_reg=7,
    random_strength=1,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    random_seed=42,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    early_stopping_rounds=50,
    verbose=50
)

model.fit(train_pool, eval_set=val_pool)

0:	test: 0.6270258	best: 0.6270258 (0)	total: 47.3ms	remaining: 14.1s
50:	test: 0.6467901	best: 0.6468755 (49)	total: 2.48s	remaining: 12.1s
100:	test: 0.6459642	best: 0.6471751 (66)	total: 3.92s	remaining: 7.72s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6471751486
bestIteration = 66

Shrink model to first 67 iterations.


CatBoostClassifier(auto_class_weights='Balanced', bootstrap_type='Bernoulli', depth=5, early_stopping_rounds=50, eval_metric='AUC', iterations=300, l2_leaf_reg=7, learning_rate=0.0001, random_seed=42, random_strength=1, subsample=0.8, verbose=50)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_pred_proba_cb = model.predict_proba(X_val)[:, 1]
roc_cb = roc_auc_score(y_val, y_pred_proba_cb)
pr_cb = average_precision_score(y_val, y_pred_proba_cb)

print(f"CatBoost: ROC-AUC={roc_cb:.4f}, PR-AUC={pr_cb:.4f}")

# Feature importance
importances = model.get_feature_importance(prettified=True)
print(importances)

CatBoost: ROC-AUC=0.6472, PR-AUC=0.2365
                  Feature Id  Importances
0           difficulty_match    43.198566
1                common_tags    25.077817
2               jaccard_tags    23.584435
3   course_popularity_weight     3.996200
4            cosine_distance     1.164350
5          course_tags_count     0.850607
6            user_tags_count     0.419821
7             user_likes_30d     0.408172
8              user_views_7d     0.327049
9              user_likes_7d     0.310765
10       user_unique_tags_7d     0.248004
11            user_views_30d     0.214406
12    user_unique_courses_7d     0.199808


#CatBoost c PairLogit функцией потерь

In [ ]:
from catboost import CatBoostRanker, Pool

train_df = train_df.sort_values(['user_temp_id', 'timestamp']).reset_index(drop=True)
val_df = val_df.sort_values(['user_temp_id', 'timestamp']).reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']

train_pool = Pool(
    X_train,
    y_train,
    group_id=train_df['user_temp_id']
)

val_pool = Pool(
    X_val,
    y_val,
    group_id=val_df['user_temp_id']
)

model = CatBoostRanker(
    loss_function='PairLogit',
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=42,
    eval_metric='MAP:top=10',
    early_stopping_rounds=50,
    verbose=50,
    custom_metric=['AUC', 'MAP:top=10', 'NDCG:top=10']
)

model.fit(train_pool, eval_set=val_pool)

0:	learn: 0.0526348	test: 0.0866395	best: 0.0866395 (0)	total: 395ms	remaining: 3m 16s
50:	learn: 0.1789137	test: 0.1486199	best: 0.1532282 (44)	total: 23.7s	remaining: 3m 28s
100:	learn: 0.1996379	test: 0.1529693	best: 0.1550277 (96)	total: 45s	remaining: 2m 57s
150:	learn: 0.2192572	test: 0.1561304	best: 0.1568264 (126)	total: 1m 8s	remaining: 2m 39s
200:	learn: 0.2353952	test: 0.1582857	best: 0.1593653 (164)	total: 1m 29s	remaining: 2m 13s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1593653189
bestIteration = 164

Shrink model to first 165 iterations.


CatBoostRanker(custom_metric=['AUC', 'MAP:top=10', 'NDCG:top=10'], depth=6, early_stopping_rounds=50, eval_metric='MAP:top=10', iterations=500, l2_leaf_reg=5, learning_rate=0.03, loss_function='PairLogit', random_seed=42, verbose=50)

## Сохранение модели для проекта

In [ ]:

import os
import pickle
from datetime import datetime


MODEL_DIR = "/content/drive/MyDrive/coursearch/model_export"
os.makedirs(MODEL_DIR, exist_ok=True)

model.save_model(os.path.join(MODEL_DIR, "catboost_model.cbm"))

model_metadata = {
    'model_type': 'CatBoostRanker',
    'loss_function': 'PairLogit',
    'feature_cols': feature_cols,
    'training_date': datetime.now().isoformat(),
    'roc_auc': float(roc_cb) if 'roc_cb' in dir() else None,
    'pr_auc': float(pr_cb) if 'pr_cb' in dir() else None,
    'iterations': model.get_params().get('iterations'),
    'learning_rate': model.get_params().get('learning_rate'),
    'depth': model.get_params().get('depth')
}

with open(os.path.join(MODEL_DIR, "model_metadata.json"), "w") as f:
    json.dump(model_metadata, f, indent=2)

if 'scaler' in dir():
    with open(os.path.join(MODEL_DIR, "scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)

with open(os.path.join(MODEL_DIR, "feature_cols.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)

inference_code = """
import json
import pickle
import pandas as pd
import numpy as np
from catboost import CatBoostRanker

# Загрузка модели
model = CatBoostRanker()
model.load_model("catboost_model.cbm")

# Загрузка признаков
with open("feature_cols.json", "r") as f:
    feature_cols = json.load(f)

# Функция для предсказания
def predict_relevance(user_features_df, course_features_df):
    '''
    user_features_df: DataFrame с признаками пользователя
    course_features_df: DataFrame с признаками курсов
    '''
    # Объединяем признаки
    X = pd.merge(user_features_df, course_features_df, how='cross')
    X = X[feature_cols]
    return model.predict(X)

print("Модель успешно загружена!")
"""

with open(os.path.join(MODEL_DIR, "inference_example.py"), "w") as f:
    f.write(inference_code)

print(f"Модель сохранена в: {MODEL_DIR}")
print("Файлы в папке:")
for f in os.listdir(MODEL_DIR):
    print(f"  - {f}")

Модель сохранена в: /content/drive/MyDrive/coursearch/model_export
Файлы в папке:
  - catboost_model.cbm
  - model_metadata.json
  - feature_cols.json
  - inference_example.py


In [ ]:

y_pred_scores = model.predict(X_val)
roc_cb = roc_auc_score(y_val, y_pred_scores)
pr_cb = average_precision_score(y_val, y_pred_scores)

print(f"CatBoost Ranker: ROC-AUC={roc_cb:.4f}, PR-AUC={pr_cb:.4f}")

importances = model.get_feature_importance(data=train_pool, prettified=True)
print(importances)

CatBoost Ranker: ROC-AUC=0.6391, PR-AUC=0.2270
                  Feature Id  Importances
0           difficulty_match     0.021815
1               jaccard_tags     0.009386
2                common_tags     0.008321
3   course_popularity_weight     0.007620
4            cosine_distance     0.004754
5          course_tags_count     0.002472
6             user_views_30d     0.002211
7              user_likes_7d     0.002206
8             user_likes_30d     0.001831
9              user_views_7d     0.001392
10    user_unique_courses_7d     0.001286
11       user_unique_tags_7d     0.001026
12           user_tags_count     0.000920


#Логистическая регрессия

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_proba_lr = lr.predict_proba(X_val_scaled)[:, 1]
roc_lr = roc_auc_score(y_val, y_pred_proba_lr)
pr_lr = average_precision_score(y_val, y_pred_proba_lr)

print(f"LogReg: ROC-AUC={roc_lr:.4f}, PR-AUC={pr_lr:.4f}")

LogReg: ROC-AUC=0.6356, PR-AUC=0.2294
